# 04 メゾ分析: タグ共起ネットワーク

各質問が持つタグのペアを共起エッジとしてカウントし、
言語ごとのタグ共起グラフを構築する。

- **Louvain 法**でコミュニティ（学習領域）を自動検出
- **媒介中心性**で「複数領域にまたがるキータグ」を特定

**出力ファイル**
- `data/analysis/network_centrality.parquet` — ノード指標テーブル
- `web/static/data/networks/{lang}.json` — D3.js 用ネットワークデータ（言語別）

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config.settings import PROCESSED_DIR, ANALYSIS_DIR, ROOT_DIR
from src.analysis.network import NetworkAnalyzer

LANGUAGES = ['python', 'javascript', 'java', 'go']
print('設定完了')

## 1. データ読み込み

In [ ]:
questions_df = pd.read_parquet(PROCESSED_DIR / 'questions.parquet')
print(f'質問数: {len(questions_df):,}')

## 2. ネットワーク構築とコミュニティ検出

In [ ]:
analyzer = NetworkAnalyzer()
graphs = {}
partitions = {}
centralities = {}

for lang in LANGUAGES:
    G = analyzer.build_cooccurrence_graph(questions_df, language=lang)
    partition = analyzer.detect_communities(G)
    centrality = analyzer.compute_centrality(G)
    
    graphs[lang] = G
    partitions[lang] = partition
    centralities[lang] = centrality
    
    print(f'{lang}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, '
          f'{len(set(partition.values()))} communities')

## 3. 媒介中心性トップ20（言語別）

In [ ]:
for lang in LANGUAGES:
    top = analyzer.top_nodes_by_centrality(centralities[lang], top_n=10)
    print(f'\n【{lang}】媒介中心性トップ10')
    for tag, score in top:
        comm = partitions[lang].get(tag, -1)
        print(f'  {tag:<30} {score:.4f}  (community {comm})')

In [ ]:
# 媒介中心性を可視化（Python の例）
lang = 'python'
top20 = analyzer.top_nodes_by_centrality(centralities[lang], top_n=20)
tags, scores = zip(*top20)

fig = px.bar(
    x=list(scores), y=list(tags),
    orientation='h',
    title=f'[{lang}] タグ媒介中心性 トップ20（学習領域の結節点）',
    labels={'x': '媒介中心性', 'y': 'タグ'},
    color=list(scores),
    color_continuous_scale='Blues',
)
fig.update_layout(yaxis=dict(autorange='reversed'), showlegend=False)
fig.show()

In [ ]:
# Go の媒介中心性
lang = 'go'
top20 = analyzer.top_nodes_by_centrality(centralities[lang], top_n=20)
tags, scores = zip(*top20)

fig2 = px.bar(
    x=list(scores), y=list(tags),
    orientation='h',
    title=f'[{lang}] タグ媒介中心性 トップ20',
    labels={'x': '媒介中心性', 'y': 'タグ'},
    color=list(scores),
    color_continuous_scale='Greens',
)
fig2.update_layout(yaxis=dict(autorange='reversed'), showlegend=False)
fig2.show()

## 4. コミュニティ別タグ分布

In [ ]:
# Python コミュニティの内訳を表示
lang = 'python'
G = graphs[lang]
partition = partitions[lang]
centrality = centralities[lang]

from collections import defaultdict
communities: dict[int, list] = defaultdict(list)
for node, comm_id in partition.items():
    freq = G.nodes[node].get('frequency', 0)
    communities[comm_id].append((node, freq))

print(f'[{lang}] コミュニティ内訳:')
for comm_id in sorted(communities):
    members = sorted(communities[comm_id], key=lambda x: -x[1])
    top_members = ', '.join(f'{t}({n})' for t, n in members[:5])
    print(f'  Community {comm_id} ({len(members)} tags): {top_members}')

In [ ]:
# コミュニティサイズの比較（全言語）
records = []
for lang in LANGUAGES:
    partition = partitions[lang]
    from collections import Counter
    comm_sizes = Counter(partition.values())
    for comm_id, size in comm_sizes.items():
        records.append({'language': lang, 'community': comm_id, 'size': size})

comm_df = pd.DataFrame(records)
fig3 = px.bar(
    comm_df, x='language', y='size', color='community',
    title='言語別コミュニティ構成（タグ数）',
    labels={'size': 'タグ数', 'language': '言語'},
)
fig3.show()

## 5. D3.js 用 JSON 確認

In [ ]:
import json

for lang in LANGUAGES:
    path = ROOT_DIR / 'web' / 'static' / 'data' / 'networks' / f'{lang}.json'
    with open(path) as f:
        data = json.load(f)
    print(f'{lang}: {len(data["nodes"])} nodes, {len(data["links"])} links → {path.name}')

## 6. 考察

### Python
- **python-3.x** が最高の媒介中心性（0.49）→ バージョン移行の質問が複数領域を横断
- **django / numpy / pandas** がそれぞれ独立したコミュニティの中核
- **web-scraping** コミュニティはフロントエンドとデータ収集の接点

### JavaScript
- **html** が最高の媒介中心性（0.33）→ DOM 操作が全コミュニティと連携
- **reactjs / node.js** が大きな独立コミュニティ（現代フロントエンド vs サーバーサイド）

### Java
- **spring-boot / java** の2タグが同程度の中心性 → エンタープライズとコアの2極構造
- **android** コミュニティが大きく、モバイル開発の重要性を反映

### Go
- **go** タグ単独の媒介中心性が0.79と突出 → コミュニティ間の橋渡しを一手に担う
- **goroutine / multithreading** が独立したコミュニティ → 並行処理が Go の特徴的な学習領域